In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [3]:
import pandas as pd
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load data
urllib.request.urlretrieve(
  'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv',
  'train.csv')
df = pd.read_csv('train.csv')

# Feature engineering
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex'] = df['Sex'].map({'female':1, 'male':0})
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FareBand'] = pd.qcut(df['Fare'], 4, labels=[0,1,2,3]).astype(int)
df['Embarked'] = df['Embarked'].map({'S':0,'C':1,'Q':2})
df = df.drop(['Cabin','Name','Ticket','PassengerId','SibSp','Parch'], axis=1)

# Split
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Train
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Accuracy: 0.7933


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest: {rf_acc:.4f}")

Random Forest: 0.8324


In [5]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_acc = accuracy_score(y_test, xgb.predict(X_test))
print(f"XGBoost: {xgb_acc:.4f}")

XGBoost: 0.8268


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
import numpy as np

# 1 — define which columns are numeric
num_cols = ['Age', 'Fare', 'Pclass', 'FamilySize']
cat_cols = ['Sex', 'IsAlone', 'FareBand', 'Embarked']

# 2 — preprocessing steps for numeric columns
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3 — combine numeric + categorical into one preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols)
])

# 4 — full pipeline: preprocessing + model
pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('model', XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_estimators=100
    ))
])

# 5 — 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline, X, y,
    cv=cv,
    scoring='roc_auc'
)

# 6 — print results
print("CV AUC scores:", np.round(scores, 4))
print(f"Mean AUC : {scores.mean():.4f}")
print(f"Std AUC  : {scores.std():.4f}")
print(f"\nIf Mean AUC > 0.85 → strong model")
print(f"If Std  AUC > 0.03 → unstable, needs tuning")


CV AUC scores: [0.9051 0.8988 0.836  0.8613 0.8813]
Mean AUC : 0.8765
Std AUC  : 0.0253

If Mean AUC > 0.85 → strong model
If Std  AUC > 0.03 → unstable, needs tuning
